# 香港 AQHI：全历史数据读取与质量检查

本 Notebook 是 `01_data_check.ipynb` 的全历史升级版，原版保留。
只读取 `data/raw/aqhi_history`，避免重复导入外面的六月样本。原始文件不修改。

按顺序运行，或使用 **Run → Run All Cells**。读取与导出需要一些时间。
本阶段不填补缺失值、不删除替代记录、不训练模型。

来源：[香港环保署历史 AQHI](https://data.gov.hk/sc-data/dataset/hk-epd-airteam-past-record-of-air-quality-health-index-sc)。
下载清单及官方数据字典位于 `data/download_records`。


## 1. 环境与文件范围
默认使用所有已下载月份；可修改起止月份，例如 `202001`。

In [13]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src' / 'aqhi_data.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'aqhi_data.py').exists(), '请从 Math7005 项目目录打开 Notebook'
sys.path.insert(0, str(ROOT / 'src'))
from aqhi_data import read_history, read_network

START_MONTH = None  # 例如 '202001'；None 表示不限制
END_MONTH = None    # 例如 '202606'
RAW = ROOT / 'data' / 'raw' / 'aqhi_history'
files = sorted(p for p in RAW.glob('*/*_ChS.csv')
               if (START_MONTH is None or p.name[:6] >= START_MONTH)
               and (END_MONTH is None or p.name[:6] <= END_MONTH))
assert files, '没有找到月度文件，请检查下载目录或月份设置'
inventory = pd.DataFrame({'file': [str(p.relative_to(ROOT)) for p in files],
                          'month': [p.name[:6] for p in files],
                          'bytes': [p.stat().st_size for p in files]})
assert not inventory.month.duplicated().any(), '同一个月份存在多个文件，需先核查'
months = pd.PeriodIndex(pd.to_datetime(inventory.month, format='%Y%m'), freq='M')
missing_months = pd.period_range(months.min(), months.max(), freq='M').difference(months).astype(str).tolist()
print(f'文件数：{len(files)}；月份：{inventory.month.min()} 至 {inventory.month.max()}')
print('范围内缺少的月份：', missing_months)
display(pd.concat([inventory.head(3), inventory.tail(3)]).drop_duplicates())

文件数：151；月份：201312 至 202606
范围内缺少的月份： []


,file,month,bytes
0,data\raw\aqhi_history\2013\201312_ChS.csv,201312,2566
1,data\raw\aqhi_history\2014\201401_ChS.csv,201401,28188
2,data\raw\aqhi_history\2014\201402_ChS.csv,201402,25245
148,data\raw\aqhi_history\2026\202604_ChS.csv,202604,31904
149,data\raw\aqhi_history\2026\202605_ChS.csv,202605,32890
150,data\raw\aqhi_history\2026\202606_ChS.csv,202606,32553


## 2. 批量读取与原始值保留

只前向填充 CSV 中省略的日期标签，不填充 AQHI。每日最高值摘要不当作小时记录。
`*` 保留为替代标记；`10+` 保留为超过上限的类别，不伪造为精确数值 11。
沿用 `24时 → 次日00:00` 的解析约定，并保留源日期和小时。时间语义仍需结合官方定义核实。
任何文件解析失败都会停止，不会静默跳过。

In [14]:
frames, file_rows = [], []
for i, path in enumerate(files, 1):
    try:
        frame = read_history(path)
    except Exception as exc:
        raise RuntimeError(f'读取失败：{path.name}；请检查源格式') from exc
    frame['source_month'] = path.name[:6]
    assert frame.source_date.str[:7].str.replace('-', '').eq(path.name[:6]).all(), '源日期与文件月份不一致'
    file_rows.append({'month': path.name[:6], 'file': path.name,
                      'rows': len(frame), 'stations': frame.station_id.nunique(),
                      'first_timestamp': str(frame.timestamp.min()),
                      'last_timestamp': str(frame.timestamp.max()),
                      'excluded_daily_summary_rows': frame.attrs.get('excluded_daily_summary_rows', 0),
                      'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    frames.append(frame)
    if i % 25 == 0 or i == len(files):
        print(f'已读取 {i}/{len(files)} 个文件')
history = pd.concat(frames, ignore_index=True).sort_values(['station_id', 'timestamp']).reset_index(drop=True)
del frames, frame
file_audit = pd.DataFrame(file_rows)
print('行数、列数：', history.shape)
display(history.head())
display(file_audit.tail())

已读取 25/151 个文件
已读取 50/151 个文件
已读取 75/151 个文件
已读取 100/151 个文件
已读取 125/151 个文件
已读取 150/151 个文件
已读取 151/151 个文件
行数、列数： (1838616, 15)


,source_date,source_hour,timestamp,station_source,station_id,aqhi_raw,aqhi_value,is_substituted,is_10_plus,is_missing,is_unavailable,is_invalid,risk_category,source_file,source_month
0,2013-12-30,01,2013-12-30 01:00:00+08:00,铜锣湾,Causeway Bay,8,8.0,False,False,False,False,False,Very High,201312_ChS.csv,201312
1,2013-12-30,02,2013-12-30 02:00:00+08:00,铜锣湾,Causeway Bay,7,7.0,False,False,False,False,False,High,201312_ChS.csv,201312
2,2013-12-30,03,2013-12-30 03:00:00+08:00,铜锣湾,Causeway Bay,6,6.0,False,False,False,False,False,Moderate,201312_ChS.csv,201312
3,2013-12-30,04,2013-12-30 04:00:00+08:00,铜锣湾,Causeway Bay,6,6.0,False,False,False,False,False,Moderate,201312_ChS.csv,201312
4,2013-12-30,05,2013-12-30 05:00:00+08:00,铜锣湾,Causeway Bay,6,6.0,False,False,False,False,False,Moderate,201312_ChS.csv,201312


,month,file,rows,stations,first_timestamp,last_timestamp,excluded_daily_summary_rows,sha256
146,202602,202602_ChS.csv,12096,18,2026-02-01 01:00:00+08:00,2026-03-01 00:00:00+08:00,28,d9f15b1d0ee59be05db70308bd06d340270d65e6044f06...
147,202603,202603_ChS.csv,13392,18,2026-03-01 01:00:00+08:00,2026-04-01 00:00:00+08:00,31,6d4b086dab473c562d42fe273f6a61e10af84e990294bc...
148,202604,202604_ChS.csv,12960,18,2026-04-01 01:00:00+08:00,2026-05-01 00:00:00+08:00,30,8ed63302ef24165a7953997fffb3ef285a1bf46ba960e0...
149,202605,202605_ChS.csv,13392,18,2026-05-01 01:00:00+08:00,2026-06-01 00:00:00+08:00,31,e766846399db7a8ebb22f4a29f68fb4f7f5ca037a5acef...
150,202606,202606_ChS.csv,12960,18,2026-06-01 01:00:00+08:00,2026-07-01 00:00:00+08:00,30,84d394496271429ae0243cd5ad12f57b78e8ed17397b75...


## 3. 重复、缺失与特殊值

标记可重叠，不能直接相加。`exact_numeric_missing` 包括 `10+`，不等于没有观测。
替代标记的具体适用规则仍应核对官方说明。重复记录只报告，不自动删除。

In [9]:
FLAGS = ['is_missing', 'is_unavailable', 'is_substituted', 'is_10_plus', 'is_invalid']
duplicate_mask = history.duplicated(['station_id', 'timestamp'], keep=False)
duplicates = history.loc[duplicate_mask].copy()
station_quality = history.groupby('station_id')[FLAGS].sum()
station_quality['rows'] = history.groupby('station_id').size()
station_quality['substituted_pct'] = (100 * station_quality.is_substituted / station_quality.rows).round(2)
station_quality['exact_numeric_missing'] = history.aqhi_value.isna().groupby(history.station_id).sum()
monthly_quality = history.groupby('source_month')[FLAGS].sum()
monthly_quality['rows'] = history.groupby('source_month').size()
print('额外重复行数：', int(history.duplicated(['station_id', 'timestamp']).sum()))
display(station_quality)
display(history.loc[history.is_invalid, ['source_file', 'station_id', 'timestamp', 'aqhi_raw']].head(20))
display(duplicates.head(10))

额外重复行数： 0


,is_missing,is_unavailable,is_substituted,is_10_plus,is_invalid,rows,substituted_pct,exact_numeric_missing
station_id,,,,,,,,
Causeway Bay,551,10,4361,152,0,109416,3.99,713
Central,524,27,2749,97,0,109416,2.51,648
Central/Western,518,8,2829,120,0,109416,2.59,646
Eastern,519,8,3946,59,0,109416,3.61,586
Kwai Chung,530,8,3194,101,0,109416,2.92,639
Kwun Tong,540,8,2542,93,0,109416,2.32,641
Mong Kok,538,9,3661,108,0,109416,3.35,655
North,1442,12,1979,24,0,53136,3.72,1478
Sha Tin,544,12,4241,62,0,109416,3.88,618


,source_file,station_id,timestamp,aqhi_raw


,source_date,source_hour,timestamp,station_source,station_id,aqhi_raw,aqhi_value,is_substituted,is_10_plus,is_missing,is_unavailable,is_invalid,risk_category,source_file,source_month


## 4. 时间覆盖与历年站点变化

按每个站点自身的首末记录检查内部小时缺口。首条记录之前、末条记录之后的时间单独列出，不能据此判断正式启用或停用日期。
所有小时行都存在，也可能有空白 AQHI。站点月份表统计的是记录行数，不是有效观测数。
2013年12月为起始的部分月份。

In [10]:
coverage_rows, gap_rows = [], []
global_start, global_end = history.timestamp.min(), history.timestamp.max()
for station, group in history.groupby('station_id'):
    observed = pd.DatetimeIndex(group.timestamp.unique()).sort_values()
    expected = pd.date_range(observed.min(), observed.max(), freq='h')
    gaps = expected.difference(observed)
    coverage_rows.append({'station_id': station, 'first_record': str(observed.min()),
        'last_record': str(observed.max()), 'unique_hour_rows': len(observed),
        'expected_hours_within_own_span': len(expected), 'missing_hours_within_own_span': len(gaps),
        'hours_before_first_record': int((observed.min()-global_start)/pd.Timedelta(hours=1)),
        'hours_after_last_record': int((global_end-observed.max())/pd.Timedelta(hours=1))})
    gap_rows.extend({'station_id': station, 'timestamp': str(t)} for t in gaps)
coverage = pd.DataFrame(coverage_rows)
hour_gaps = pd.DataFrame(gap_rows, columns=['station_id', 'timestamp'])
station_month_rows = pd.crosstab(history.source_month, history.station_id)
station_count = station_month_rows.gt(0).sum(axis=1)
station_changes = station_count.loc[station_count.ne(station_count.shift())].rename('station_count').to_frame()
display(coverage)
print('站点数量发生变化的月份（按文件内站点列统计）：')
display(station_changes)
print('内部小时缺口示例：')
display(hour_gaps.head(20))

,station_id,first_record,last_record,unique_hour_rows,expected_hours_within_own_span,missing_hours_within_own_span,hours_before_first_record,hours_after_last_record
0,Causeway Bay,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
1,Central,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
2,Central/Western,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
3,Eastern,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
4,Kwai Chung,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
5,Kwun Tong,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
6,Mong Kok,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
7,North,2020-06-01 01:00:00+08:00,2026-07-01 00:00:00+08:00,53136,53304,168,56280,0
8,Sha Tin,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0
9,Sham Shui Po,2013-12-30 01:00:00+08:00,2026-07-01 00:00:00+08:00,109416,109584,168,0,0


站点数量发生变化的月份（按文件内站点列统计）：


,station_count
source_month,
201312,15
201602,16
202006,18


内部小时缺口示例：


,station_id,timestamp
0,Causeway Bay,2023-08-25 01:00:00+08:00
1,Causeway Bay,2023-08-25 02:00:00+08:00
2,Causeway Bay,2023-08-25 03:00:00+08:00
3,Causeway Bay,2023-08-25 04:00:00+08:00
4,Causeway Bay,2023-08-25 05:00:00+08:00
5,Causeway Bay,2023-08-25 06:00:00+08:00
6,Causeway Bay,2023-08-25 07:00:00+08:00
7,Causeway Bay,2023-08-25 08:00:00+08:00
8,Causeway Bay,2023-08-25 09:00:00+08:00
9,Causeway Bay,2023-08-25 10:00:00+08:00


## 5. 匹配站点资料并汇总

地理资料是现有下载快照，只用于站点标识与地图参考；不能证明站点历年位置未改变。
本 Notebook 不把 RSS 快照混入历史训练数据。

In [11]:
network = read_network(ROOT / 'data' / 'raw' / 'Air_Quality_Monitoring_Network_of_Hong_Kong_GEOJSON.zip')
unmatched_stations = sorted(set(history.station_id) - set(network.station_id))
summary = {'files': len(files), 'rows': len(history), 'stations': int(history.station_id.nunique()),
    'source_month_start': inventory.month.min(), 'source_month_end': inventory.month.max(),
    'timestamp_start': str(global_start), 'timestamp_end': str(global_end),
    'missing_months': missing_months,
    'excluded_daily_summary_rows': int(file_audit.excluded_daily_summary_rows.sum()),
    'duplicate_extra_rows': int(history.duplicated(['station_id', 'timestamp']).sum()),
    'missing_hours_within_station_spans': len(hour_gaps),
    'unmatched_stations': unmatched_stations,
    'quality_flags': {f: int(history[f].sum()) for f in FLAGS},
    'timestamp_convention': 'source_date + source_hour hours; 24 = next-day 00:00; verify source semantics',
    'imputation': 'none', 'model_ready': False}
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(network[['station_id', 'station_type', 'longitude', 'latitude']])

{
  "files": 151,
  "rows": 1838616,
  "stations": 18,
  "source_month_start": "201312",
  "source_month_end": "202606",
  "timestamp_start": "2013-12-30 01:00:00+08:00",
  "timestamp_end": "2026-07-01 00:00:00+08:00",
  "missing_months": [],
  "excluded_daily_summary_rows": 4559,
  "duplicate_extra_rows": 0,
  "missing_hours_within_station_spans": 3024,
  "unmatched_stations": [],
  "quality_flags": {
    "is_missing": 16424,
    "is_unavailable": 1413,
    "is_substituted": 63177,
    "is_10_plus": 1772,
    "is_invalid": 0
  },
  "timestamp_convention": "source_date + source_hour hours; 24 = next-day 00:00; verify source semantics",
  "imputation": "none",
  "model_ready": false
}


,station_id,station_type,longitude,latitude
0,Tsuen Wan,GENERAL MONITORING STATION,114.114535,22.371742
1,Causeway Bay,ROADSIDE MONITORING STATION,114.185090,22.280133
2,Central,ROADSIDE MONITORING STATION,114.158127,22.281815
3,Central/Western,GENERAL MONITORING STATION,114.144421,22.284891
4,Mong Kok,ROADSIDE MONITORING STATION,114.168272,22.322611
5,Tai Po,GENERAL MONITORING STATION,114.164570,22.450960
6,Sham Shui Po,GENERAL MONITORING STATION,114.159109,22.330226
7,Sha Tin,GENERAL MONITORING STATION,114.184532,22.376281
8,Yuen Long,GENERAL MONITORING STATION,114.022649,22.445155
9,Kwai Chung,GENERAL MONITORING STATION,114.129601,22.357104


## 6. 导出全历史整理结果

每次运行创建独立时间戳目录，不覆盖既有结果。小时明细使用压缩 CSV，避免生成庞大未压缩文件。
站点快照单独保存，通过 `station_id` 关联。下一阶段先审查质量报告，再选择建模范围与按时间划分训练/验证/测试集。
**导出成功不代表数据已经适合直接训练。**

In [12]:
from datetime import datetime
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = ROOT / 'data' / 'processed' / 'full_history' / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
history.to_csv(OUT / 'history_all_audited.csv.gz', index=False, encoding='utf-8-sig', compression='gzip')
for name, table, keep_index in [
    ('file_audit', file_audit, False), ('station_quality', station_quality, True),
    ('monthly_quality', monthly_quality, True), ('station_coverage', coverage, False),
    ('station_month_rows', station_month_rows, True), ('station_changes', station_changes, True),
    ('hour_gaps', hour_gaps, False), ('duplicates', duplicates, False), ('stations_snapshot', network, False)]:
    table.to_csv(OUT / f'{name}.csv', index=keep_index, encoding='utf-8-sig')
(OUT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
assert len(history) == int(file_audit.rows.sum()), '导出前行数核对失败'
print('已导出全历史数据与检查报告：', OUT)
print('原始文件与六月样本结果均未修改。')

已导出全历史数据与检查报告： C:\Users\hxq08259\Desktop\sem1\Math7005\data\processed\full_history\20261110_170756_651230
原始文件与六月样本结果均未修改。


In [15]:
# 1. 查看每个站点缺口的起止时间和数量
gap_check = hour_gaps.copy()
gap_check["timestamp"] = pd.to_datetime(gap_check["timestamp"], utc=True)

display(
    gap_check.groupby("station_id")["timestamp"]
    .agg(开始="min", 结束="max", 缺少小时数="count")
)

# 2. 检查缺口是否为所有18个站点共同缺失
counts = gap_check.groupby("timestamp")["station_id"].nunique()
print("缺口时间点总数：", len(counts))
print("是否每个缺口时间都缺少18个站点：", counts.eq(18).all())

# 3. 直接读取2023年8月原始文件，列出源文件中的日期
import csv

path = RAW / "2023" / "202308_ChS.csv"
with path.open(encoding="utf-8-sig", newline="") as f:
    raw_rows = list(csv.reader(f))

dates = sorted({
    row[0].strip()
    for row in raw_rows
    if row and row[0].strip().startswith("2023-08-")
})
print("原始文件实际包含的日期：")
print(dates)

,开始,结束,缺少小时数
station_id,,,
Causeway Bay,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Central,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Central/Western,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Eastern,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Kwai Chung,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Kwun Tong,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Mong Kok,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
North,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168
Sha Tin,2023-08-24 17:00:00+00:00,2023-08-31 16:00:00+00:00,168


缺口时间点总数： 168
是否每个缺口时间都缺少18个站点： True
原始文件实际包含的日期：
['2023-08-01', '2023-08-02', '2023-08-03', '2023-08-04', '2023-08-05', '2023-08-06', '2023-08-07', '2023-08-08', '2023-08-09', '2023-08-10', '2023-08-11', '2023-08-12', '2023-08-13', '2023-08-14', '2023-08-15', '2023-08-16', '2023-08-17', '2023-08-18', '2023-08-19', '2023-08-20', '2023-08-21', '2023-08-22', '2023-08-23', '2023-08-24']
